# End-to-End Pipeline Integration Test

This notebook loads the unified `TerraFluxPipeline` and tests it against sample rows to ensure all 3 models work in harmony.

In [26]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import json
import sys
import os
import importlib

# Add the workspace root to Python's path so it can import from 'src'
sys.path.append(os.path.abspath("../../"))

import src.utils
import src.pipeline
importlib.reload(src.utils)
importlib.reload(src.pipeline)

from src.pipeline import TerraFluxPipeline

# Initialize the unified pipeline
# Note: This will load all preprocessor artifacts and model weights
pipeline = TerraFluxPipeline()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\weights\esa_soc_model.pkl
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\preprocessors\esa_scaler.pkl
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\preprocessors\esa_pca.pkl
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\weights\m2_rural_risk_model.pkl
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\preprocessors\rural_scaler.pkl
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\weights\m3_srdb_regression_model.pkl
Model loaded <- C:\Users\justi\Downloads\ML project data\Terraflux\models\preprocessors\m3_srdb_features.pkl


### Prepare Sample Data
Extract one sample row from each of the raw datasets to simulate a real-world API request.

In [27]:
# Load ESA SOC data
esa_df = pd.read_csv("../../data/raw/esa_eo4_test_soc.csv")
esa_sample = esa_df.drop(columns=["logoc_d.f"], errors="ignore").iloc[0:1]

# Load Rural Risk data
rural_df = pd.read_csv("../../data/raw/rural_carbon_dataset1.csv")
rural_sample = rural_df.drop(columns=["Risk"], errors="ignore").iloc[0:1]

# Define a sample biome for SRDB
biome_sample = "Temperate Broadleaf Forest"

print("ESA Sample Shape:", esa_sample.shape)
print("Rural Sample Shape:", rural_sample.shape)

ESA Sample Shape: (1, 226)
Rural Sample Shape: (1, 13)


### Run the Unified Prediction
Pass the multiple data sources into `evaluate_region`.

In [28]:
# Execute the end-to-end pipeline
result = pipeline.evaluate_region(
    esa_data=esa_sample,
    rural_data=rural_sample,
    biome=biome_sample
)

# Safely print the final combined ESG/Carbon assessment as JSON
print(json.dumps(result, indent=4))

{
    "predicted_soc_log": [
        2.059473752975464
    ],
    "predicted_emission_risk": [
        "Low"
    ],
    "predicted_annual_soil_respiration_gC_m2": [
        1137.4615478515625
    ]
}
